## 1. Imports, Parameters, and Local Paths

In [1]:
from pathlib import Path
import re, json, hashlib, platform
import numpy as np
import pandas as pd
import scipy, sklearn, joblib
from scipy import sparse
from IPython.display import display
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.cluster import MiniBatchKMeans
from sklearn.linear_model import LinearRegression

In [151]:
# Set this to the last month confirmed complete, for example "2026-04".
last_complete_month = None

# Compare training lengths from 3 to 24 months, increasing by one month.
WINDOWS = list(range(3, 25))

# Move the validation cutoff forward one month at a time.
N_VALID_MONTHS = 3

LOWER_QUANTILE = 0.005
UPPER_QUANTILE = 0.995

# Drop nonessential categorical features with more than 80% missing values.
MISSING_LIMIT = 0.80

# Group categories appearing fewer than 20 times into an infrequent category.
MIN_CATEGORY_COUNT = 20

# Limit each categorical feature to at most 100 encoded columns.
MAX_CATEGORIES = 100

# Require at least 5 historical sales for ZIP- or county-level median Price Per Square Foot (PPSF).
MIN_COMPS = 5

# Create up to 20 geographic clusters using training coordinates.
N_CLUSTERS = 20

# Keep these True to retain the two deletion rules in current notebook.
# Set False to keep these records with review flags instead.
DROP_CONTRACT_BEFORE_LISTING = True
DROP_ROOM_AREA_ANOMALIES = True

SEED = 42
np.random.seed(SEED)

In [153]:
data_dir = Path("/Users/jingying/Documents/IDX Exchange/py/csv")
result_dir = Path("cleaned_data")

files = []
for file in data_dir.glob("CRMLSSold*.csv"):
    if re.fullmatch(r"CRMLSSold\d{6}(?:_filled)?.csv", file.name):
        files.append(file)       
files.sort(key=lambda file: file.name[9:15])

if not files:
    raise FileNotFoundError(f"No monthly CSV files found in {data_dir}")

# Create a new run path; keep result_dir as your original output root.
run_id = pd.Timestamp.now(tz="UTC").strftime("%Y%m%d_%H%M%S_%f")
run_dir = result_dir / ("run_" + run_id)

## 2. Load Records and Normalize Column Types

In [29]:
monthly_dfs, manifest = [], []

for file in files:
    monthly_data = pd.read_csv(file, low_memory=False)

    # Record the source file and record position.
    monthly_data["source_file"] = file.name
    # Adding 2 makes the first data record correspond to CSV line 2, because line 1 contains the column names.
    monthly_data["source_row"] = np.arange(len(monthly_data)) + 2
    monthly_dfs.append(monthly_data)

    # Record the row count and file checksum.
    with file.open("rb") as f:
        manifest.append({
            "file": file.name,
            "rows": len(monthly_data),
            "sha256": hashlib.file_digest(f, "sha256").hexdigest()
        })
    
combined_sold_data = pd.concat(monthly_dfs, ignore_index=True)
combined_sold_data["row_id"] = np.arange(len(combined_sold_data))

print(f"Read {len(files)} files，combined {len(combined_sold_data):,} rows.")
display(pd.DataFrame(manifest).drop(columns="sha256"))

Read 28 files，combined 615,707 rows.


,file,rows
0,CRMLSSold202401_filled.csv,17958
1,CRMLSSold202402.csv,19925
2,CRMLSSold202403_filled.csv,23276
3,CRMLSSold202404_filled.csv,24640
4,CRMLSSold202405_filled.csv,26487
5,CRMLSSold202406_filled.csv,24328
6,CRMLSSold202407_filled.csv,26240
7,CRMLSSold202408.csv,24558
8,CRMLSSold202409.csv,21267
9,CRMLSSold202410.csv,23274


In [32]:
base_num = ["LivingArea", "BedroomsTotal", "BathroomsTotalInteger",
            "LotSizeSquareFeet", "YearBuilt", "Latitude", "Longitude"]
base_cat = ["PostalCode", "City", "CountyOrParish", "UnparsedAddress", "SubdivisionName",
            "ElementarySchoolDistrict", "MiddleOrJuniorSchoolDistrict", "HighSchoolDistrict"]
base_date = ["CloseDate", "ListingContractDate", "PurchaseContractDate"]

raw = combined_sold_data.copy()

for col in ["ClosePrice"] + base_num:
    raw[col] = pd.to_numeric(raw[col], errors="coerce").replace([np.inf, -np.inf], np.nan)
    
for col in base_date:
    raw[col] = pd.to_datetime(raw[col], errors="coerce", format="mixed").dt.normalize()
    
for col in base_cat:
    raw[col] = (raw[col].astype("string")
                .str.strip()
                .str.lower()
                .str.replace(r"\s+", " ", regex=True)
                .replace("", pd.NA))

# Store the identifier as a string.
raw["ListingKey"] = raw["ListingKey"].astype("string")

# Standardize ZIP codes to five digits.    
raw["PostalCode"] = raw["PostalCode"].str.extract(r"^(\d{5})(?:-\d{4}|\.0)?$", expand=False)

## 3. Fixed Cleaning Rules, Repairs, and Review Flags

### 3.1 Row-removal log

In [34]:
data = raw.copy()

# Store cleaning summaries, removed records, and repair summaries.
clean_log, removed_parts, repair_log = [], [], []

# Record the original number of rows before any cleaning steps.
initial_n = len(data)

def remove_rows(df, mask, step):
    """
    Remove rows that satisfy a condition and record the cleaning details.

    Parameters:
    df (pandas.DataFrame): The DataFrame to clean.
    mask (boolean Series or array-like): True = remove the row; False = keep the row.
    step (str): A description of the cleaning rule.

    Returns: (pandas.DataFrame) A copy of the DataFrame containing only the retained rows.
    """
    
    mask = pd.Series(mask, index=df.index).fillna(False).astype(bool)

    # Count rows marked True, which will be removed.
    n_mask = int(mask.sum())

    # Save a summary of this cleaning step.
    clean_log.append({
        "step": step,              # Name of the cleaning rule.
        "before": len(df),         # Number of rows before this cleaning step.
        "removed": n_mask,         # Number of rows removed.
        "after": len(df) - n_mask, # Number of rows remaining.

        # Percentage removed relative to this step's input.
        "pct_step%": 100 * n_mask / len(df),

        # Percentage removed in this step relative to the original dataset.
        "pct_original%": 100 * n_mask / initial_n
    })

    # Save the identifiers and source information of removed rows.
    # Add a column explaining why these rows were removed.
    removed_parts.append(
        df.loc[mask, ["row_id", "source_file", "source_row"]]
        .assign(reason=step)
    )

    return df.loc[~mask].copy()
  

In [37]:
data = remove_rows(data, ~((data["PropertyType"] == "Residential") & (data["PropertySubType"] == "SingleFamilyResidence")), "Restrict to Resi. & SingleFamily")
data = remove_rows(data, (data["ClosePrice"].isna()) | (data["ClosePrice"]<=0), "ClosePrice is NA or zero or negative")
data = remove_rows(data, data["CloseDate"] < data["ListingContractDate"], "CloseDate earlier than ListingDate")
data = remove_rows(data, data["PurchaseContractDate"] < data["ListingContractDate"], "PurchaseDate earlier than ListingDate")
data = remove_rows(data, (data["LivingArea"] <= 0) | (data["LotSizeSquareFeet"] <= 0), "zero or negative square footage")
data = remove_rows(data, (data["BedroomsTotal"] * 100 > data["LivingArea"]) | (data["BathroomsTotalInteger"] * 50 > data["LivingArea"]), "beds/baths counts inconsistent with living area")

In [40]:
#Duplicate listings for the same property/transaction (same parcel ID or address + close date).
keys = ["UnparsedAddress", "City", "CloseDate"]
data = remove_rows(data, (data[keys].notna().all(axis=1)) & (data.duplicated(keys, keep="first")), "duplicated address+city+closedate")

In [42]:
#Duplicate listings for the same ListingKey
data = remove_rows(data, (data["ListingKey"].notna()) & (data.duplicated(["ListingKey", "CloseDate"], keep="first")), "duplicated ListingKey+closedate")
data["address_missing_flag"] = data[["UnparsedAddress", "City"]].isna().any(axis=1).astype(int)

In [61]:
display(pd.DataFrame(clean_log).round(4))
print("Rows after data cleaning: ", len(data))

,step,before,removed,after,pct_step%,pct_original%
0,Restrict to Resi. & SingleFamily,615707,305972,309735,49.6944,49.6944
1,ClosePrice is NA or zero or negative,309735,3,309732,0.0010,0.0005
2,CloseDate earlier than ListingDate,309732,44,309688,0.0142,0.0071
3,PurchaseDate earlier than ListingDate,309688,145,309543,0.0468,0.0236
4,zero or negative square footage,309543,287,309256,0.0927,0.0466
5,beds/baths counts inconsistent with living area,309256,8,309248,0.0026,0.0013
6,duplicated address+city+closedate,309248,171,309077,0.0553,0.0278
7,duplicated ListingKey+closedate,309077,0,309077,0.0000,0.0000


Rows after data cleaning:  309077


In [63]:
#removed_df = pd.concat(removed_parts, ignore_index=True)

# Display removed records, excluding the property-type restriction.
#display(removed_df.loc[removed_df["reason"] != "Restrict to Resi. & SingleFamily"])

## 4. Define the Test Month and Rolling Validation Splits

In [90]:
# Convert closing dates to calendar months.
data["CloseMonth"] = data["CloseDate"].dt.to_period("M")
latest_month = data["CloseMonth"].max()

# Choose the final test month.
if last_complete_month:
    test_month = pd.Period(last_complete_month, freq="M")
else:
    test_month = latest_month
    print("Confirm that this latest observed month is complete:", test_month)

if test_month not in data["CloseMonth"].values:
    raise ValueError("Test month is not in the cleaned data.")

# Keep a separate analysis dataset ending at the test month.
future_rows = (data["CloseMonth"] > test_month).sum()
print("Records excluded after the test month:", future_rows)

# Preserve data so changing test_month and rerunning is safe.
model_data = data.loc[data["CloseMonth"] <= test_month].copy()

# Reserve this month for final evaluation, not window selection.
test_data = model_data.loc[model_data["CloseMonth"] == test_month].copy()

Confirm that this latest observed month is complete: 2026-04
Records excluded after the test month: 0


In [94]:
# Count the number of records in each observed month.
month_counts = model_data.groupby("CloseMonth").size()
month_counts.name = "rows"
#print(month_counts)

# Store the months that have at least one record.
available_months = month_counts.index
print(available_months)

# Show record counts for the latest eight observed months.
#display(month_counts.tail(8).to_frame())

PeriodIndex(['2024-01', '2024-02', '2024-03', '2024-04', '2024-05', '2024-06',
             '2024-07', '2024-08', '2024-09', '2024-10', '2024-11', '2024-12',
             '2025-01', '2025-02', '2025-03', '2025-04', '2025-05', '2025-06',
             '2025-07', '2025-08', '2025-09', '2025-10', '2025-11', '2025-12',
             '2026-01', '2026-02', '2026-03', '2026-04'],
            dtype='period[M]', name='CloseMonth')


In [98]:
# 1. Choose three validation months before the test month.
valid_months = []

for i in range(N_VALID_MONTHS, 0, -1):
    valid_months.append(test_month - i)

# Example: test month = 2026-04
# Validation months = 2026-01, 2026-02, 2026-03


# 2. Check that each validation month has data.
for month in valid_months:
    if month not in available_months:
        raise ValueError(f"No data for validation month: {month}")


# 3. Check each candidate training window.
# WINDOWS contains the training lengths we want to compare.
# Example: WINDOWS = [3, 6, 9, 12]

valid_windows = []
skipped_windows = []

for x in WINDOWS:
    enough_history = True

    # Check the training history needed for each validation month.
    for month in valid_months:
        train_start = month - x
        train_end = month - 1

        # List every calendar month required for this training period.
        required_months = pd.period_range(start=train_start, end=train_end, freq="M")

        # Reject this window if any required month has no data.
        for train_month in required_months:
            if train_month not in available_months:
                enough_history = False
                break

        if not enough_history:
            break

    # Keep only windows that work for all validation months.
    if enough_history:
        valid_windows.append(x)
    else:
        skipped_windows.append(x)


# Stop if none of the candidate windows has enough history.
if len(valid_windows) == 0:
    raise ValueError("Not enough history. Try smaller values in WINDOWS.")


# 4. Display the selected months and eligible training windows.
print("Test month:", test_month)

print("Validation months:")
for month in valid_months:
    print(" ", month)

print("Available training windows:", valid_windows)
print("Skipped training windows:", skipped_windows)

Test month: 2026-04
Validation months:
  2026-01
  2026-02
  2026-03
Available training windows: [3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24]
Skipped training windows: []


In [100]:
# Display each eligible window and its three validation splits.
# This creates a split plan only; it does not fit a model or select the best X.
split_rows = []

for x in valid_windows:
    for month in valid_months:
        train_start = month - x
        train_end = month - 1
        train_mask = model_data["CloseMonth"].between(train_start, train_end)
        valid_mask = model_data["CloseMonth"] == month

        split_rows.append({
            "window_months": x,
            "train_start": str(train_start),
            "train_end": str(train_end),
            "validation_month": str(month),
            "train_rows": int(train_mask.sum()),
            "validation_rows": int(valid_mask.sum())
        })

split_plan = pd.DataFrame(split_rows)
display(split_plan)
print("Reserved test rows:", len(test_data))

,window_months,train_start,train_end,validation_month,train_rows,validation_rows
0,3,2025-10,2025-12,2026-01,32171,7470
1,3,2025-11,2026-01,2026-02,27633,8940
2,3,2025-12,2026-02,2026-03,26844,11546
3,4,2025-09,2025-12,2026-01,43603,7470
4,4,2025-10,2026-01,2026-02,39641,8940
5,4,2025-11,2026-02,2026-03,36573,11546
6,5,2025-08,2025-12,2026-01,55026,7470
7,5,2025-09,2026-01,2026-02,51073,8940
8,5,2025-10,2026-02,2026-03,48581,11546
9,6,2025-07,2025-12,2026-01,67114,7470


Reserved test rows: 12458


## 5. Feature Engineering and Information Timing

In [175]:
num_features = base_num + ["PropertyAge", "BedBathRatio", "MonthSin", "MonthCos", "TimeIndex"]

# base_cat includes the full address for cleaning, but the model must exclude it.
model_cat = [col for col in base_cat if col != "UnparsedAddress"]
cat_features = model_cat + ["GeoCluster"]

history_features = ["HistoricPPSF"]
flag_features = [c + "_missing" for c in num_features + history_features]

In [177]:
def basic_features(df):
    """Create row-level features without learning population statistics."""
    f = df.copy()
    f["ValuationDate"] = f["CloseMonth"].dt.to_timestamp()
    f["PropertyAge"] = f["ValuationDate"].dt.year - f["YearBuilt"]
    f["BedBathRatio"] = f["BedroomsTotal"] / f["BathroomsTotalInteger"].replace(0, np.nan)
    month = f["ValuationDate"].dt.month

    # Encode the month as a position on a yearly cycle.
    f["MonthSin"] = np.sin(2*np.pi*month/12)
    f["MonthCos"] = np.cos(2*np.pi*month/12)

    # Count months since January 2020 to represent a long-term time trend.
    f["TimeIndex"] = (f["ValuationDate"].dt.year-2020)*12 + month-1
    
    for c in base_cat:
        f[c] = f[c].astype(object).where(f[c].notna(), np.nan)
    return f

def add_history(rows, reference):
    """Use earlier-month training sales only; reference excludes evaluation records."""
    out = rows.copy()
    #Price Per Square Foot
    out["HistoricPPSF"] = np.nan
    
    ref = reference.copy()
    # Derive the reference price per square foot from training sales only.
    ref["PPSF"] = ref["ClosePrice"] / ref["LivingArea"]
    for month in out["CloseMonth"].unique():
        # For this valuation month, exclude the same month and all future months.
        hist = ref.loc[(ref["CloseMonth"] >= month-12) & (ref["CloseMonth"] < month)
                       & np.isfinite(ref.PPSF) & ref["PPSF"].gt(0)]
        idx = out.index[out["CloseMonth"].eq(month)]
        if hist.empty:
            continue
        value = pd.Series(np.nan, index=idx, dtype=float)
        # Try ZIP first, then county; fill only values still missing.
        for group in ["PostalCode", "CountyOrParish"]:
            stats = hist.groupby(group)["PPSF"].agg(["median", "count"])
            medians = stats.loc[stats["count"] >= MIN_COMPS, "median"]
            value = value.fillna(out.loc[idx, group].map(medians))
        # Use the overall earlier-sales median for any remaining unmatched rows.
        out.loc[idx, "HistoricPPSF"] = value.fillna(hist["PPSF"].median())
    return out

# With no earlier history, use missing now and fixed zero plus a flag later.
def add_missing_flags(frame):
    frame = frame.copy()
    for c in num_features + history_features:
        frame[c+"_missing"] = frame[c].isna().astype(int)
    return frame

### 5.1 Geographic Clusters

In [133]:
def fit_geo(train):
    """Fit a location model using complete coordinate pairs from training rows."""
    coords = train[["Latitude", "Longitude"]].dropna()
    k = min(N_CLUSTERS, len(coords.drop_duplicates()))
    if k == 0:
        return None
    geo = Pipeline([("scale", StandardScaler()),
                    ("cluster", MiniBatchKMeans(n_clusters=k, random_state=SEED, n_init=3, batch_size=2048))])
    return geo.fit(coords)

def add_geo(frame, geo):
    """Apply the fitted location model; keep incomplete locations as unknown."""
    frame = frame.copy()
    frame["GeoCluster"] = "unknown"
    ok = frame[["Latitude", "Longitude"]].notna().all(axis=1)
    if geo is not None and ok.any():
        frame.loc[ok, "GeoCluster"] = geo.predict(frame.loc[ok, ["Latitude", "Longitude"]]).astype(str)
    return frame

## 6. Training-Only Imputation, Encoding, and Scaling

In [135]:
def make_preprocessor(train):
    """Choose columns from training missingness and return an unfitted transformer."""
    # Learn missingness and column-selection decisions from this training fold only.
    keep_num = [c for c in num_features if train[c].notna().any()]
    essential = ["PostalCode", "City", "CountyOrParish", "GeoCluster"]
    keep_cat = [c for c in cat_features if train[c].notna().any() and
                (c in essential or train[c].isna().mean() <= MISSING_LIMIT)]
    numeric = Pipeline([("impute", SimpleImputer(strategy="median")), 
                        ("scale", StandardScaler())])
    categorical = Pipeline([
        ("impute", SimpleImputer(strategy="constant", fill_value="__missing__")),
        ("encode", OneHotEncoder(handle_unknown="infrequent_if_exist",
                                 min_frequency=MIN_CATEGORY_COUNT, max_categories=MAX_CATEGORIES,
                                 sparse_output=True))])
    history = Pipeline([("impute", SimpleImputer(strategy="constant", fill_value=0, keep_empty_features=True)),
                        ("scale", StandardScaler())])
    pre = ColumnTransformer([
        ("num", numeric, keep_num), ("cat", categorical, keep_cat),
        ("history", history, history_features), ("flags", "passthrough", flag_features)
    ], sparse_threshold=1.0)
    schema = {"numeric": keep_num, 
              "categorical": keep_cat,
              "dropped": sorted(set(num_features+cat_features)-set(keep_num+keep_cat))}
    return pre, schema

## 7. Fit One Training Window and Transform Later Records

In [137]:
def metrics(y, pred):
    """Calculate errors for strictly positive targets; preserve empty-scope results."""
    y = np.asarray(y, dtype=float)
    pred = np.asarray(pred, dtype=float)
    if len(y) != len(pred):
        raise ValueError("Labels and predictions have different lengths.")
    if len(y) == 0:
        return {"n": 0, "MAPE_pct": np.nan, "MdAPE_pct": np.nan, "MAE": np.nan}
    if not np.isfinite(y).all() or not np.isfinite(pred).all() or (y <= 0).any():
        raise ValueError("Invalid labels or predictions supplied to metrics.")
    absolute_error = np.abs(y - pred)
    ape = 100 * absolute_error / y
    return {"n": len(y), "MAPE_pct": float(ape.mean()),
            "MdAPE_pct": float(np.median(ape)), "MAE": float(absolute_error.mean())}

In [139]:
def fit_window(train_raw):
    """Fit price bounds, feature models, preprocessing, and regression on training only."""
    low, high = train_raw.ClosePrice.quantile([LOWER_QUANTILE, UPPER_QUANTILE])
    train = train_raw.loc[train_raw.ClosePrice.between(low, high)].copy()
    if len(train) < 2:
        raise ValueError("Fewer than two training rows remain after price filtering.")
    reference = train_raw[["CloseMonth", "ClosePrice", "LivingArea",
                           "PostalCode", "CountyOrParish"]].copy()
    train = add_history(basic_features(train), reference)
    geo = fit_geo(train)
    train = add_missing_flags(add_geo(train, geo))
    pre, schema = make_preprocessor(train)
    feature_cols = schema["numeric"] + schema["categorical"] + history_features + flag_features
    model = Pipeline([("preprocess", pre), ("regression", LinearRegression())])
    # Guard the explicit feature list before fitting.
    forbidden = {"ClosePrice", "ListPrice", "OriginalListPrice", "DaysOnMarket",
                 "CloseDate", "PurchaseContractDate", "ListingContractDate", "row_id",
                 "UnparsedAddress", "ListingKey", "source_file", "source_row",
                 "contract_before_listing_flag", "contract_date_review_flag"}
    if forbidden.intersection(feature_cols):
        raise ValueError("A forbidden field entered the feature list.")
    model.fit(train[feature_cols], train.ClosePrice)
    bundle = {"pipeline": model, "geo": geo, "reference": reference,
              "schema": schema, "feature_cols": feature_cols,
              "price_bounds": [float(low), float(high)]}
    return bundle, train

def transform_rows(rows, bundle):
    # The reference contains training sales only; evaluation targets are not used.
    frame = add_history(basic_features(rows), bundle["reference"])
    return add_missing_flags(add_geo(frame, bundle["geo"]))

## 8. Run Rolling Validation for Every Eligible X

In [141]:
fold_rows, outlier_logs, schema_rows, price_exclusions = [], [], [], []
for x in valid_windows:
    for v in valid_months:
        # Training ends before the first day of the validation month.
        train_raw = model_data.loc[model_data.CloseMonth.between(v - x, v - 1)].copy()
        valid_raw = model_data.loc[model_data.CloseMonth.eq(v)].copy()
        assert train_raw.CloseDate.max() < valid_raw.CloseDate.min()
        assert set(train_raw.row_id).isdisjoint(valid_raw.row_id)
        bundle, train = fit_window(train_raw)
        valid = transform_rows(valid_raw, bundle)
        pred = bundle["pipeline"].predict(valid[bundle["feature_cols"]])
        low, high = bundle["price_bounds"]
        in_range = valid.ClosePrice.between(low, high).to_numpy()

        # Record row exclusions and feature choices for this specific fold.
        excluded = train_raw.loc[~train_raw.ClosePrice.between(low, high), ["row_id"]]
        price_exclusions.append(excluded.assign(stage="validation", X=x, month=str(v)))
        schema_rows.append({"X": x, "month": str(v), **bundle["schema"]})
        outlier_logs.append({"stage": "validation", "X": x, "month": str(v),
                            "low": low, "high": high, "train_before": len(train_raw),
                            "train_after": len(train), "eval_before": len(valid),
                            "eval_in_range": int(in_range.sum())})

        # Use the same complete month for window selection across all X values.
        for scope, mask in [("all", np.ones(len(valid), dtype=bool)),
                            ("avm_range", in_range)]:
            fold_rows.append({"X": x, "month": str(v), "scope": scope,
                              "negative_predictions": int((pred[mask] < 0).sum()),
                              **metrics(valid.ClosePrice.to_numpy()[mask], pred[mask])})
        print(f"Finished X={x:2d}, validation={v}, train={len(train):,}, validation={len(valid):,}")

fold_scores = pd.DataFrame(fold_rows)

Finished X= 3, validation=2026-01, train=31,854, validation=7,470
Finished X= 3, validation=2026-02, train=27,359, validation=8,940
Finished X= 3, validation=2026-03, train=26,574, validation=11,546
Finished X= 4, validation=2026-01, train=43,170, validation=7,470
Finished X= 4, validation=2026-02, train=39,246, validation=8,940
Finished X= 4, validation=2026-03, train=36,217, validation=11,546
Finished X= 5, validation=2026-01, train=54,481, validation=7,470
Finished X= 5, validation=2026-02, train=50,569, validation=8,940
Finished X= 5, validation=2026-03, train=48,099, validation=11,546
Finished X= 6, validation=2026-01, train=66,456, validation=7,470
Finished X= 6, validation=2026-02, train=61,878, validation=8,940
Finished X= 6, validation=2026-03, train=59,412, validation=11,546
Finished X= 7, validation=2026-01, train=78,008, validation=7,470
Finished X= 7, validation=2026-02, train=73,844, validation=8,940
Finished X= 7, validation=2026-03, train=70,723, validation=11,546
Finis

## 9. Compare Validation Results and Select X

In [143]:
# Use only the full-month validation rows for selection.
full_scores = fold_scores.loc[fold_scores["scope"] == "all"].copy()
if not np.isfinite(full_scores["MdAPE_pct"]).all():
    raise ValueError("A full validation fold has an invalid MdAPE. Review the failed fold.")

window_scores = full_scores.groupby("X", as_index=False).agg(
    mean_MdAPE=("MdAPE_pct", "mean"),
    sd_MdAPE=("MdAPE_pct", "std"),
    mean_MAPE=("MAPE_pct", "mean"),
    mean_MAE=("MAE", "mean"),
    folds=("month", "nunique")
)
if not window_scores["folds"].eq(N_VALID_MONTHS).all():
    raise ValueError("Candidate windows were not evaluated on all validation months.")
window_scores = window_scores.sort_values(["mean_MdAPE", "X"]).reset_index(drop=True)
best_x = int(window_scores.loc[0, "X"])

print("Selected training length:", best_x, "months")
display(window_scores.round(3))
display(fold_scores.round(3))

# A compact month-by-window table helps reveal unstable performance.
monthly_comparison = full_scores.pivot(index="X", columns="month", values="MdAPE_pct")
display(monthly_comparison.round(3))

Selected training length: 24 months


,X,mean_MdAPE,sd_MdAPE,mean_MAPE,mean_MAE,folds
0,24,18.177,0.438,4087.558,340151.444,3
1,23,18.257,0.429,4107.739,341079.393,3
2,22,18.406,0.578,4114.972,342555.063,3
3,21,18.421,0.524,4160.885,343242.007,3
4,19,18.620,0.493,4127.363,344350.676,3
5,20,18.671,0.617,4141.051,344716.894,3
6,18,18.680,0.548,4132.375,344124.704,3
7,17,18.954,0.685,4172.420,346269.029,3
8,16,18.960,0.635,4180.784,346311.553,3
9,15,18.983,0.796,4142.376,346479.608,3


,X,month,scope,negative_predictions,n,MAPE_pct,MdAPE_pct,MAE
0,3,2026-01,all,8,7470,13998.046,28.576,469723.598
1,3,2026-01,avm_range,5,7380,37.411,28.105,360024.722
2,3,2026-02,all,3,8940,51.609,39.879,482622.345
3,3,2026-02,avm_range,3,8864,49.189,39.614,453203.268
4,3,2026-03,all,8,11546,36.688,26.503,458532.649
5,3,2026-03,avm_range,6,11418,35.408,26.180,354872.889
6,4,2026-01,all,159,7470,11718.220,22.776,403374.241
7,4,2026-01,avm_range,145,7378,31.103,22.484,291128.447
8,4,2026-02,all,96,8940,30.612,21.216,328732.637
9,4,2026-02,avm_range,81,8862,29.202,20.999,295990.193


month,2026-01,2026-02,2026-03
X,,,
3,28.576,39.879,26.503
4,22.776,21.216,20.708
5,22.326,20.326,20.237
6,21.514,22.651,20.081
7,21.048,21.114,19.581
8,21.100,20.601,19.315
9,20.312,20.231,18.869
10,19.987,19.552,18.927
11,19.853,19.210,18.541


## 10. Refit on the Selected Final Training Window

In [145]:
train_raw = model_data.loc[model_data.CloseMonth.between(test_month - best_x, test_month - 1)].copy()
test_raw = model_data.loc[model_data.CloseMonth.eq(test_month)].copy()
final_bundle, train_final = fit_window(train_raw)
test_final = transform_rows(test_raw, final_bundle)
low, high = final_bundle["price_bounds"]
pre = final_bundle["pipeline"].named_steps["preprocess"]
feature_cols = final_bundle["feature_cols"]

# Transform only: test data never enters fit or fit_transform.
X_train = sparse.csr_matrix(pre.transform(train_final[feature_cols]))
X_test = sparse.csr_matrix(pre.transform(test_final[feature_cols]))
y_train = train_final.ClosePrice
y_test = test_final.ClosePrice
in_test_range = test_final.ClosePrice.between(low, high).to_numpy()

assert train_final.CloseDate.max() < test_final.CloseDate.min()
assert set(train_final.row_id).isdisjoint(test_final.row_id)
assert X_train.shape[1] == X_test.shape[1]
assert len(y_train) == X_train.shape[0] and len(y_test) == X_test.shape[0]
assert np.isfinite(X_train.data).all() and np.isfinite(X_test.data).all()

outlier_logs.append({"stage": "final", "X": best_x, "month": str(test_month),
                    "low": low, "high": high, "train_before": len(train_raw),
                    "train_after": len(train_final), "eval_before": len(test_final),
                    "eval_in_range": int(in_test_range.sum())})
excluded = train_raw.loc[~train_raw.ClosePrice.between(low, high), ["row_id"]]
price_exclusions.append(excluded.assign(stage="final", X=best_x, month=str(test_month)))

print("Final training months:", test_month - best_x, "through", test_month - 1)
print("Test month:", test_month)
print("Matrix shapes:", X_train.shape, X_test.shape)
print("Test predictions and test scores have not been calculated.")

Final training months: 2024-04 through 2026-03
Test month: 2026-04
Matrix shapes: (264531, 497) (12458, 497)
Test predictions and test scores have not been calculated.


### 10.1 Inspect Training Missingness and Learned Parameters

In [146]:
missing_report = pd.DataFrame({
    "before_price_filter_pct": train_raw[base_num + model_cat].isna().mean() * 100,
    "retained_training_pct": train_final[base_num + model_cat].isna().mean() * 100
})
missing_report["action"] = ["Training median and missing flag" if c in base_num
                            else "Missing category and one-hot encoding" for c in missing_report.index]
dropped = final_bundle["schema"]["dropped"]
missing_report.loc[missing_report.index.isin(dropped), "action"] = "Dropped using training missingness"
display(missing_report.round(2))
print("Dropped inputs:", dropped)

# Show the actual medians and scaling parameters learned on final training rows.
nums = final_bundle["schema"]["numeric"]
if nums:
    numeric_steps = pre.named_transformers_["num"].named_steps
    numeric_parameters = pd.DataFrame({
        "feature": nums, "fill_value": numeric_steps["impute"].statistics_,
        "mean_after_imputation": numeric_steps["scale"].mean_,
        "scale_after_imputation": numeric_steps["scale"].scale_
    })
else:
    numeric_parameters = pd.DataFrame(columns=["feature", "fill_value", "mean_after_imputation", "scale_after_imputation"])
display(numeric_parameters)

# Group-level missingness is descriptive and uses training records only.
city_missing = train_raw.assign(lot_missing=train_raw.LotSizeSquareFeet.isna()).groupby("City", dropna=False).agg(n=("row_id", "size"), lot_missing_rate=("lot_missing", "mean"))
display(city_missing.loc[city_missing.n >= 100].sort_values("lot_missing_rate", ascending=False).head(10))

,before_price_filter_pct,retained_training_pct,action
LivingArea,0.05,0.04,Training median and missing flag
BedroomsTotal,0.00,0.00,Training median and missing flag
BathroomsTotalInteger,0.02,0.02,Training median and missing flag
LotSizeSquareFeet,1.69,1.70,Training median and missing flag
YearBuilt,0.06,0.05,Training median and missing flag
Latitude,0.07,0.07,Training median and missing flag
Longitude,0.07,0.07,Training median and missing flag
PostalCode,0.01,0.01,Missing category and one-hot encoding
City,0.08,0.07,Missing category and one-hot encoding
CountyOrParish,0.00,0.00,Missing category and one-hot encoding


Dropped inputs: ['ElementarySchoolDistrict', 'MiddleOrJuniorSchoolDistrict']


,feature,fill_value,mean_after_imputation,scale_after_imputation
0,LivingArea,1.804000e+03,2015.487358,9.395861e+02
1,BedroomsTotal,3.000000e+00,3.480333,9.470332e-01
2,BathroomsTotalInteger,2.000000e+00,2.600043,1.067849e+00
3,LotSizeSquareFeet,7.230000e+03,293025.513565,1.557774e+07
4,YearBuilt,1.976000e+03,1975.553859,2.742304e+01
5,Latitude,3.409446e+01,34.788672,1.769859e+00
6,Longitude,-1.180507e+02,-118.677861,3.132976e+00
7,PropertyAge,4.900000e+01,49.141341,2.741887e+01
8,BedBathRatio,1.500000e+00,1.462432,4.736129e-01
9,MonthSin,-2.449294e-16,-0.028390,7.077748e-01


,n,lot_missing_rate
City,,
coronado,232,0.344828
imperial beach,173,0.225434
la jolla,522,0.210728
san diego,8941,0.187563
lemon grove,252,0.186508
alpine,279,0.186380
lakeside,432,0.175926
poway,638,0.175549
bonita,188,0.164894


## 11. Export Week 3 Deliverables

In [155]:
run_dir.mkdir(parents=True, exist_ok=False)

def readable_csv(frame, split):
    """Create readable, imputed rows using the final training fit; do not refit."""
    schema = final_bundle["schema"]
    out = frame[["row_id", "source_file", "source_row", "CloseDate", "ValuationDate"] + feature_cols].copy()
    nums, cats = schema["numeric"], schema["categorical"]
    if nums:
        out[nums] = pre.named_transformers_["num"].named_steps["impute"].transform(frame[nums])
    if cats:
        out[cats] = pre.named_transformers_["cat"].named_steps["impute"].transform(frame[cats])
    out[history_features] = pre.named_transformers_["history"].named_steps["impute"].transform(frame[history_features])
    out["ClosePrice"] = frame.ClosePrice
    out["split"] = split
    out["in_train_price_range"] = frame.ClosePrice.between(low, high)
    return out

train_csv = readable_csv(train_final, "train")
test_csv = readable_csv(test_final, "test")
cleaned = pd.concat([train_csv, test_csv], ignore_index=True)
for name, frame in {"cleaned_data": cleaned, "train_cleaned": train_csv,
                    "test_cleaned": test_csv, "test_avm_range": test_csv.loc[in_test_range],
                    "cleaning_log": pd.DataFrame(clean_log),
                    "removed_rows": pd.concat(removed_parts, ignore_index=True),
                    "repair_log": pd.DataFrame(repair_log), "missing_report": missing_report.reset_index(),
                    "window_scores": window_scores, "fold_scores": fold_scores,
                    "data_manifest": pd.DataFrame(manifest)}.items():
    frame.to_csv(run_dir / f"{name}.csv", index=False)

### 11.1 Split, Cleaning, and Price-Filtering Audits

In [165]:
price_audit = pd.DataFrame(outlier_logs)
price_audit["train_removed"] = price_audit.train_before - price_audit.train_after
price_audit["train_removed_pct"] = 100 * price_audit.train_removed / price_audit.train_before
price_audit["eval_outside_pct"] = 100 * (1 - price_audit.eval_in_range / price_audit.eval_before)
price_audit.to_csv(run_dir / "price_filter_log.csv", index=False)
pd.concat(price_exclusions, ignore_index=True).to_csv(run_dir / "price_exclusions_by_fold.csv", index=False)
split_plan.to_csv(run_dir / "split_plan.csv", index=False)
monthly_comparison.to_csv(run_dir / "monthly_validation_MdAPE.csv")
month_counts.to_csv(run_dir / "monthly_counts.csv")
numeric_parameters.to_csv(run_dir / "numeric_parameters.csv", index=False)
city_missing.to_csv(run_dir / "training_city_missingness.csv")

# Export only needed history fields, not unrelated agent/contact columns.
history_columns = ["row_id", "source_file", "source_row", "ListingKey", "UnparsedAddress",
                   "CloseDate", "CloseMonth", "ClosePrice"] + base_num + base_cat
history_columns = list(dict.fromkeys(history_columns))
model_data[history_columns].to_csv(run_dir / "quality_cleaned_history.csv", index=False)
with (run_dir / "validation_schemas.json").open("w", encoding="utf-8") as f:
    json.dump(schema_rows, f, indent=2)
display(price_audit.round(3))
print("Cleaning logs, split summaries, and cleaned history were saved.")

,stage,X,month,low,high,train_before,train_after,eval_before,eval_in_range,train_removed,train_removed_pct,eval_outside_pct
0,validation,3,2026-01,180000.000,8750000.00,32171,31854,7470,7380,317,0.985,1.205
1,validation,3,2026-02,180000.000,9178897.76,27633,27359,8940,8864,274,0.992,0.850
2,validation,3,2026-03,179215.000,8900650.00,26844,26574,11546,11418,270,1.006,1.109
3,validation,4,2026-01,185000.000,8749775.00,43603,43170,7470,7378,433,0.993,1.232
4,validation,4,2026-02,180000.000,9035771.20,39641,39246,8940,8862,395,0.996,0.872
5,validation,4,2026-03,180000.000,9000000.00,36573,36217,11546,11422,356,0.973,1.074
6,validation,5,2026-01,185000.000,8750000.00,55026,54481,7470,7379,545,0.990,1.218
7,validation,5,2026-02,185000.000,9000000.00,51073,50569,8940,8858,504,0.987,0.917
8,validation,5,2026-03,180000.000,8921750.00,48581,48099,11546,11418,482,0.992,1.109
9,validation,6,2026-01,185000.000,8750000.00,67114,66456,7470,7379,658,0.980,1.218


Cleaning logs, split summaries, and cleaned history were saved.


### 11.2 Save Matrices, Fitted Objects, and Reproducibility Settings

In [167]:
sparse.save_npz(run_dir / "X_train.npz", X_train)
sparse.save_npz(run_dir / "X_test.npz", X_test)
sparse.save_npz(run_dir / "X_test_avm_range.npz", X_test[in_test_range])
train_final[["row_id", "ClosePrice"]].to_csv(run_dir / "y_train.csv", index=False)
test_final[["row_id", "ClosePrice"]].to_csv(run_dir / "y_test.csv", index=False)
test_final.loc[in_test_range, ["row_id", "ClosePrice"]].to_csv(run_dir / "y_test_avm_range.csv", index=False)
pd.Series(pre.get_feature_names_out(), name="feature").to_csv(run_dir / "encoded_features.csv", index=False)
joblib.dump(final_bundle, run_dir / "preprocessing_and_baseline.joblib", compress=3)

versions = {"numpy": np.__version__, "pandas": pd.__version__, "scipy": scipy.__version__,
            "scikit-learn": sklearn.__version__, "joblib": joblib.__version__}
(run_dir / "requirements.txt").write_text("\n".join(f"{k}=={v}" for k, v in versions.items()) + "\n")
summary = {
    "test_month": str(test_month), "validation_months": [str(v) for v in valid_months],
    "requested_windows": WINDOWS, "eligible_windows": valid_windows, "skipped_windows": skipped_windows,
    "best_X": best_x, "price_bounds": [float(low), float(high)],
    "price_quantiles": [LOWER_QUANTILE, UPPER_QUANTILE],
    "selection_metric": "Mean monthly validation MdAPE on all quality-cleaned validation records",
    "tie_break": "Shorter X for exact ties", "random_seed": SEED,
    "missing_limit": MISSING_LIMIT, "minimum_comps": MIN_COMPS, "clusters": N_CLUSTERS,
    "minimum_category_count": MIN_CATEGORY_COUNT, "maximum_categories": MAX_CATEGORIES,
    "python_version": platform.python_version(), "versions": versions,
    "numeric_inputs": final_bundle["schema"]["numeric"],
    "categorical_inputs": final_bundle["schema"]["categorical"],
    "dropped_inputs": final_bundle["schema"]["dropped"],
    "feature_cols": feature_cols,
    "train_start": str(test_month - best_x), "train_end": str(test_month - 1),
    "train_rows": len(train_csv), "test_rows": len(test_csv),
    "test_avm_range_rows": int(in_test_range.sum()),
    "records_after_test_excluded": int(future_rows),
    "complete_month_assumed": last_complete_month is None,
    "last_complete_month_setting": last_complete_month,
    "drop_contract_before_listing": True,
    "drop_room_area_anomalies": True,
    "test_predictions_generated": False, "test_scored": False
}
(run_dir / "run_summary.json").write_text(json.dumps(summary, indent=2))
print("Saved Week 3(d) outputs to:", run_dir.resolve())
display(cleaned.head())

Saved Week 3(d) outputs to: /Users/jingying/Documents/IDX Exchange/py/cleaned_data/run_20261014_180636_999980


,row_id,source_file,source_row,CloseDate,ValuationDate,LivingArea,BedroomsTotal,BathroomsTotalInteger,LotSizeSquareFeet,YearBuilt,...,Longitude_missing,PropertyAge_missing,BedBathRatio_missing,MonthSin_missing,MonthCos_missing,TimeIndex_missing,HistoricPPSF_missing,ClosePrice,split,in_train_price_range
0,61161,CRMLSSold202404_filled.csv,4,2024-04-02,2024-04-01,1723.0,2.0,2.0,8100.0,1971.0,...,0,0,0,0,0,0,1,250000.0,train,True
1,61166,CRMLSSold202404_filled.csv,9,2024-04-30,2024-04-01,2285.0,3.0,3.0,43560.0,1990.0,...,0,0,0,0,0,0,1,413700.0,train,True
2,61171,CRMLSSold202404_filled.csv,14,2024-04-03,2024-04-01,1716.0,5.0,2.0,10557.0,1945.0,...,0,0,0,0,0,0,1,725000.0,train,True
3,61184,CRMLSSold202404_filled.csv,27,2024-04-12,2024-04-01,1600.0,3.0,3.0,8130.0,1958.0,...,0,0,0,0,0,0,1,600000.0,train,True
4,61186,CRMLSSold202404_filled.csv,29,2024-04-10,2024-04-01,2001.0,4.0,2.0,15600.0,1964.0,...,0,0,0,0,0,0,1,1810000.0,train,True


## 12. Reload and Verify the Exported Data

In [169]:
saved_bundle = joblib.load(run_dir / "preprocessing_and_baseline.joblib")
saved_X_train = sparse.load_npz(run_dir / "X_train.npz")
saved_X_test = sparse.load_npz(run_dir / "X_test.npz")
saved_y_train = pd.read_csv(run_dir / "y_train.csv")
saved_y_test = pd.read_csv(run_dir / "y_test.csv")
saved_names = pd.read_csv(run_dir / "encoded_features.csv")["feature"]

assert saved_X_train.shape[0] == len(saved_y_train)
assert saved_X_test.shape[0] == len(saved_y_test)
assert saved_X_train.shape[1] == saved_X_test.shape[1] == len(saved_names)
assert np.array_equal(saved_y_train.row_id.to_numpy(), train_final.row_id.to_numpy())
assert np.array_equal(saved_y_test.row_id.to_numpy(), test_final.row_id.to_numpy())
assert saved_names.tolist() == list(saved_bundle["pipeline"].named_steps["preprocess"].get_feature_names_out())
print("Export checks passed. No test predictions were generated.")

# Inspect a small numeric preview without densifying the full matrix.
display(pd.DataFrame(saved_X_train[:5, :10].toarray(), columns=saved_names.iloc[:10]).round(3))

Export checks passed. No test predictions were generated.


feature,num__LivingArea,num__BedroomsTotal,num__BathroomsTotalInteger,num__LotSizeSquareFeet,num__YearBuilt,num__Latitude,num__Longitude,num__PropertyAge,num__BedBathRatio,num__MonthSin
0,-0.311,-1.563,-0.562,-0.018,-0.166,1.127,-0.348,0.141,-0.976,1.264
1,0.287,-0.507,0.375,-0.016,0.527,-0.669,0.704,-0.552,-0.976,1.264
2,-0.319,1.605,-0.562,-0.018,-1.114,-1.262,0.520,1.089,2.191,1.264
3,-0.442,-0.507,0.375,-0.018,-0.640,-1.156,0.539,0.615,-0.976,1.264
4,-0.015,0.549,-0.562,-0.018,-0.421,1.725,-1.101,0.396,1.135,1.264


## 13. Run-Specific Results and Interpretation

In [173]:
best_row = window_scores.iloc[0]
report_lines = [
    "# Week 3 Run Results", "",
    f"Source files: {len(files)}.",
    f"Combined records: {len(combined_sold_data):,}.",
    f"Records after fixed-rule cleaning: {len(data):,}.",
    f"Analysis records through the test month: {len(model_data):,}.",
    f"Test month: {test_month}.",
    "Validation months: " + ", ".join(str(v) for v in valid_months) + ".",
    f"Selected X: {best_x} months.",
    f"Mean monthly validation MdAPE: {best_row['mean_MdAPE']:.3f}%.",
    f"Standard deviation of monthly MdAPE: {best_row['sd_MdAPE']:.3f} percentage points.",
    f"Final training period: {test_month - best_x} through {test_month - 1}.",
    f"Final retained training rows: {len(train_final):,}.",
    f"Full test rows: {len(test_final):,}.",
    f"Test rows within frozen training price bounds: {int(in_test_range.sum()):,}.",
    f"Encoded input columns: {X_train.shape[1]:,}.", "",
    "The test set was transformed and exported, but no test predictions or scores were calculated.",
    "The validation scores were used to select X and are not independent final performance estimates.",
    "Source completeness and historical availability assumptions still require confirmation."
]
report_text = "\n".join(report_lines) + "\n"
(run_dir / "run_results.md").write_text(report_text, encoding="utf-8")
print(report_text)

# Week 3 Run Results

Source files: 28.
Combined records: 615,707.
Records after fixed-rule cleaning: 309,077.
Analysis records through the test month: 309,077.
Test month: 2026-04.
Validation months: 2026-01, 2026-02, 2026-03.
Selected X: 24 months.
Mean monthly validation MdAPE: 18.177%.
Standard deviation of monthly MdAPE: 0.438 percentage points.
Final training period: 2024-04 through 2026-03.
Final retained training rows: 264,531.
Full test rows: 12,458.
Test rows within frozen training price bounds: 12,321.
Encoded input columns: 497.

The test set was transformed and exported, but no test predictions or scores were calculated.
The validation scores were used to select X and are not independent final performance estimates.
Source completeness and historical availability assumptions still require confirmation.

